In [1]:
from pyspark.sql.session import SparkSession

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .config("spark.ui.port","4141")
        .config("spark.port.maxRetries", "0")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def stop_start(app_name):
   
    global spark 
    
    if 'spark' in globals():
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
             print(f"No active session to stop. Error: {e}")
    else:
        print("No existing 'spark' variable found.")

    # 3. Return the fresh session
    print("Starting new Spark session...")
    return get_spark_session(app_name)


spark = stop_start("working_with_different_data_type")

No existing 'spark' variable found.
Starting new Spark session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/09 10:37:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
retail_df = (
    spark
    .read
    .format("csv")
    .option("inferSchema", "false")
    .option("header","true")
    .load("../data/retail-data")
)

retail_df.show(10,False)

+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|537226   |22811    |SET OF 6 T-LIGHTS CACTI       |6       |2010-12-06 08:34:00|2.95     |15987.0   |United Kingdom|
|537226   |21713    |CITRONELLA CANDLE FLOWERPOT   |8       |2010-12-06 08:34:00|2.1      |15987.0   |United Kingdom|
|537226   |22927    |GREEN GIANT GARDEN THERMOMETER|2       |2010-12-06 08:34:00|5.95     |15987.0   |United Kingdom|
|537226   |20802    |SMALL GLASS SUNDAE DISH CLEAR |6       |2010-12-06 08:34:00|1.65     |15987.0   |United Kingdom|
|537226   |22052    |VINTAGE CARAVAN GIFT WRAP     |25      |2010-12-06 08:34:00|0.42     |15987.0   |United Kingdom|
|537226   |22705    |WRAP GREEN PEARS              |25  

## Structs
You can think of structs as DataFrames within DataFrames.

In [3]:
from pyspark.sql.functions import col, struct


col_description = col("description")
col_invoiceno = col("invoiceno")



In [4]:


df = (
    retail_df
    .select(struct(col_description, col_invoiceno).alias("description_invoice"))
    
)

df.show(10,False)

df.printSchema()


+----------------------------------------+
|description_invoice                     |
+----------------------------------------+
|{SET OF 6 T-LIGHTS CACTI , 537226}      |
|{CITRONELLA CANDLE FLOWERPOT, 537226}   |
|{GREEN GIANT GARDEN THERMOMETER, 537226}|
|{SMALL GLASS SUNDAE DISH CLEAR, 537226} |
|{VINTAGE CARAVAN GIFT WRAP, 537226}     |
|{WRAP GREEN PEARS , 537226}             |
|{GOLD EAR MUFF HEADPHONES, 537226}      |
|{IVORY KNITTED MUG COSY , 537226}       |
|{PAPERWEIGHT SAVE THE PLANET, 537226}   |
|{CHRISTMAS LIGHTS 10 REINDEER, 537227}  |
+----------------------------------------+
only showing top 10 rows
root
 |-- description_invoice: struct (nullable = false)
 |    |-- description: string (nullable = true)
 |    |-- invoiceno: string (nullable = true)



In [5]:
df.select("description_invoice.description").show(5,False)

+------------------------------+
|description                   |
+------------------------------+
|SET OF 6 T-LIGHTS CACTI       |
|CITRONELLA CANDLE FLOWERPOT   |
|GREEN GIANT GARDEN THERMOMETER|
|SMALL GLASS SUNDAE DISH CLEAR |
|VINTAGE CARAVAN GIFT WRAP     |
+------------------------------+
only showing top 5 rows


In [6]:
df.select( col("description_invoice").getField("invoiceno")).show(5,False)

+-----------------------------+
|description_invoice.invoiceno|
+-----------------------------+
|537226                       |
|537226                       |
|537226                       |
|537226                       |
|537226                       |
+-----------------------------+
only showing top 5 rows


## Arrays

- To define arrays, let’s work through a use case. With our current data, our objective is to take
every single word in our city column and convert that into a row in our DataFrame.

In [7]:

from pyspark.sql.functions import split

data = [
    {"emp_id": 101, "name": "Alice", "department": "HR", "salary": 50000, "city":"Pune, Banglore, Gorakhpur"},
    {"emp_id": 102, "name": "Bob", "department": "Engineering", "salary": 85000, "city":"PrayagRaj, Kolkata, Tuscon"},
    {"emp_id": 103, "name": "Charlie", "department": "Sales", "salary": 65000, "city":"Texax, LA, New Yark"},
    {"emp_id": 104, "name": "David", "department": "Engineering", "salary": 90000, "city":"Pune, New Yark, Gorakhpur"},
    {"emp_id": 105, "name": "Charls", "department": "Cleaner", "salary": 54545, "city":"Pune, Aurangabad, None"},
    {"emp_id": 106, "name": "DeadPool", "department": "Saver", "salary": 438438748, "city": None},
]

df = spark.createDataFrame(data)
df.printSchema()
df.show(10, False)



root
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)



+--------------------------+-----------+------+--------+---------+
|city                      |department |emp_id|name    |salary   |
+--------------------------+-----------+------+--------+---------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |
|NULL                      |Saver      |106   |DeadPool|438438748|
+--------------------------+-----------+------+--------+---------+



In [8]:
array_df =(   
    df
    .withColumn("splitted_City", split(col("city"),","))
    .withColumn("index_one", col("splitted_City")[0])

)

array_df.show(10,False)

array_df.printSchema()

+--------------------------+-----------+------+--------+---------+------------------------------+---------+
|city                      |department |emp_id|name    |salary   |splitted_City                 |index_one|
+--------------------------+-----------+------+--------+---------+------------------------------+---------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |[Pune,  Banglore,  Gorakhpur] |Pune     |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |[PrayagRaj,  Kolkata,  Tuscon]|PrayagRaj|
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |[Texax,  LA,  New Yark]       |Texax    |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |[Pune,  New Yark,  Gorakhpur] |Pune     |
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |[Pune,  Aurangabad,  None]    |Pune     |
|NULL                      |Saver      |106   |DeadPool|438438748|NULL                          |NULL     |
+--------------------------+

In [9]:
from pyspark.sql.functions import size

(
    array_df
    .select(col("splitted_City"), size(col("splitted_City")))
    .show(10, False)

)

+------------------------------+-------------------+
|splitted_City                 |size(splitted_City)|
+------------------------------+-------------------+
|[Pune,  Banglore,  Gorakhpur] |3                  |
|[PrayagRaj,  Kolkata,  Tuscon]|3                  |
|[Texax,  LA,  New Yark]       |3                  |
|[Pune,  New Yark,  Gorakhpur] |3                  |
|[Pune,  Aurangabad,  None]    |3                  |
|NULL                          |NULL               |
+------------------------------+-------------------+



In [10]:
from pyspark.sql.functions import array_contains

(
    array_df
    .select(col("splitted_City"), array_contains(col("splitted_City"), "Pune"))
    .show(10, False)

)

+------------------------------+-----------------------------------+
|splitted_City                 |array_contains(splitted_City, Pune)|
+------------------------------+-----------------------------------+
|[Pune,  Banglore,  Gorakhpur] |true                               |
|[PrayagRaj,  Kolkata,  Tuscon]|false                              |
|[Texax,  LA,  New Yark]       |false                              |
|[Pune,  New Yark,  Gorakhpur] |true                               |
|[Pune,  Aurangabad,  None]    |true                               |
|NULL                          |NULL                               |
+------------------------------+-----------------------------------+



## explode

-- The explode function takes a column that consists of arrays and creates one row (with the rest of
the values duplicated) per value in the array.

- The core difference is that explode drops rows containing null or empty arrays/maps, whereas explode_outer keeps them and fills the exploded columns with null.


In [11]:
df.show(10,False)

+--------------------------+-----------+------+--------+---------+
|city                      |department |emp_id|name    |salary   |
+--------------------------+-----------+------+--------+---------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |
|NULL                      |Saver      |106   |DeadPool|438438748|
+--------------------------+-----------+------+--------+---------+



In [12]:
from pyspark.sql.functions import explode, explode_outer

splitted_df = (
  df.withColumn("splitted", split(col("city"),",") )
)

print("splitted_df:")
splitted_df.show(10,False)

print("exploded_df:")

exploded_df = (

   splitted_df
    .withColumn("exploded_city", explode(col("splitted")))
)
    

exploded_df.show(30, False)


    

splitted_df:
+--------------------------+-----------+------+--------+---------+------------------------------+
|city                      |department |emp_id|name    |salary   |splitted                      |
+--------------------------+-----------+------+--------+---------+------------------------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |[Pune,  Banglore,  Gorakhpur] |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |[PrayagRaj,  Kolkata,  Tuscon]|
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |[Texax,  LA,  New Yark]       |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |[Pune,  New Yark,  Gorakhpur] |
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |[Pune,  Aurangabad,  None]    |
|NULL                      |Saver      |106   |DeadPool|438438748|NULL                          |
+--------------------------+-----------+------+--------+---------+------------------------------+

explod

In [13]:
print("exploded_outer:")
## This is giving the city where it's either empty or None 
exploded_outer_df = (

   splitted_df
    .withColumn("exploded_city", explode_outer(col("splitted")))
)

exploded_outer_df.show(20, False)
    

exploded_outer:
+--------------------------+-----------+------+--------+---------+------------------------------+-------------+
|city                      |department |emp_id|name    |salary   |splitted                      |exploded_city|
+--------------------------+-----------+------+--------+---------+------------------------------+-------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |[Pune,  Banglore,  Gorakhpur] |Pune         |
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |[Pune,  Banglore,  Gorakhpur] | Banglore    |
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |[Pune,  Banglore,  Gorakhpur] | Gorakhpur   |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |[PrayagRaj,  Kolkata,  Tuscon]|PrayagRaj    |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |[PrayagRaj,  Kolkata,  Tuscon]| Kolkata     |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |[PrayagRaj,  Kolkata, 

In [14]:
from pyspark.sql.functions import array_agg

(exploded_outer_df.groupBy(col("city"), col("department"), col("emp_id"), col("name"), col("salary"))
  .agg(array_agg(col("exploded_city")).alias("restored_city"))
).show(10, False)

+--------------------------+-----------+------+--------+---------+------------------------------+
|city                      |department |emp_id|name    |salary   |restored_city                 |
+--------------------------+-----------+------+--------+---------+------------------------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |[Pune,  Banglore,  Gorakhpur] |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |[PrayagRaj,  Kolkata,  Tuscon]|
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |[Texax,  LA,  New Yark]       |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |[Pune,  New Yark,  Gorakhpur] |
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |[Pune,  Aurangabad,  None]    |
|NULL                      |Saver      |106   |DeadPool|438438748|[]                            |
+--------------------------+-----------+------+--------+---------+------------------------------+



## map

In [15]:
from pyspark.sql.functions import create_map

df.withColumn("created_map", create_map(col("department"), col("emp_id"))).show(10, False)

+--------------------------+-----------+------+--------+---------+--------------------+
|city                      |department |emp_id|name    |salary   |created_map         |
+--------------------------+-----------+------+--------+---------+--------------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |{HR -> 101}         |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |{Engineering -> 102}|
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |{Sales -> 103}      |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |{Engineering -> 104}|
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |{Cleaner -> 105}    |
|NULL                      |Saver      |106   |DeadPool|438438748|{Saver -> 106}      |
+--------------------------+-----------+------+--------+---------+--------------------+



In [16]:
df_map = df.withColumn("created_map", create_map(col("emp_id").cast("string"), col("department"), col("department"), col("name") ))
df_map.show(10, False)

+--------------------------+-----------+------+--------+---------+------------------------------------------+
|city                      |department |emp_id|name    |salary   |created_map                               |
+--------------------------+-----------+------+--------+---------+------------------------------------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |{101 -> HR, HR -> Alice}                  |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |{102 -> Engineering, Engineering -> Bob}  |
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |{103 -> Sales, Sales -> Charlie}          |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |{104 -> Engineering, Engineering -> David}|
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |{105 -> Cleaner, Cleaner -> Charls}       |
|NULL                      |Saver      |106   |DeadPool|438438748|{106 -> Saver, Saver -> DeadPool}         |
+---------

In [17]:
from pyspark.sql.functions import expr, map_keys, map_values, element_at

df_map.select(map_keys("created_map")).show(10, False)

+---------------------+
|map_keys(created_map)|
+---------------------+
|[101, HR]            |
|[102, Engineering]   |
|[103, Sales]         |
|[104, Engineering]   |
|[105, Cleaner]       |
|[106, Saver]         |
+---------------------+



In [18]:
df_map.select(map_values("created_map")).show(10, False)

+-----------------------+
|map_values(created_map)|
+-----------------------+
|[HR, Alice]            |
|[Engineering, Bob]     |
|[Sales, Charlie]       |
|[Engineering, David]   |
|[Cleaner, Charls]      |
|[Saver, DeadPool]      |
+-----------------------+



In [19]:
df_map.where(col("created_map")["101"] == "HR").show(10, False)

+-------------------------+----------+------+-----+------+------------------------+
|city                     |department|emp_id|name |salary|created_map             |
+-------------------------+----------+------+-----+------+------------------------+
|Pune, Banglore, Gorakhpur|HR        |101   |Alice|50000 |{101 -> HR, HR -> Alice}|
+-------------------------+----------+------+-----+------+------------------------+



In [20]:
df_map.show(10,False)

+--------------------------+-----------+------+--------+---------+------------------------------------------+
|city                      |department |emp_id|name    |salary   |created_map                               |
+--------------------------+-----------+------+--------+---------+------------------------------------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |{101 -> HR, HR -> Alice}                  |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |{102 -> Engineering, Engineering -> Bob}  |
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |{103 -> Sales, Sales -> Charlie}          |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |{104 -> Engineering, Engineering -> David}|
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |{105 -> Cleaner, Cleaner -> Charls}       |
|NULL                      |Saver      |106   |DeadPool|438438748|{106 -> Saver, Saver -> DeadPool}         |
+---------

In [21]:
df.show()

+--------------------+-----------+------+--------+---------+
|                city| department|emp_id|    name|   salary|
+--------------------+-----------+------+--------+---------+
|Pune, Banglore, G...|         HR|   101|   Alice|    50000|
|PrayagRaj, Kolkat...|Engineering|   102|     Bob|    85000|
| Texax, LA, New Yark|      Sales|   103| Charlie|    65000|
|Pune, New Yark, G...|Engineering|   104|   David|    90000|
|Pune, Aurangabad,...|    Cleaner|   105|  Charls|    54545|
|                NULL|      Saver|   106|DeadPool|438438748|
+--------------------+-----------+------+--------+---------+



In [28]:
df_single_map = (

   df.withColumn("created_map", create_map(col("emp_id"), col("department")))
)

df_single_map.show(10, False)


+--------------------------+-----------+------+--------+---------+--------------------+
|city                      |department |emp_id|name    |salary   |created_map         |
+--------------------------+-----------+------+--------+---------+--------------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |{101 -> HR}         |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |{102 -> Engineering}|
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |{103 -> Sales}      |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |{104 -> Engineering}|
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |{105 -> Cleaner}    |
|NULL                      |Saver      |106   |DeadPool|438438748|{106 -> Saver}      |
+--------------------------+-----------+------+--------+---------+--------------------+



In [31]:
df_single_map.select("created_map", explode(col("created_map")).alias("keys","values")).show(10, False)

+--------------------+----+-----------+
|created_map         |keys|values     |
+--------------------+----+-----------+
|{101 -> HR}         |101 |HR         |
|{102 -> Engineering}|102 |Engineering|
|{103 -> Sales}      |103 |Sales      |
|{104 -> Engineering}|104 |Engineering|
|{105 -> Cleaner}    |105 |Cleaner    |
|{106 -> Saver}      |106 |Saver      |
+--------------------+----+-----------+



In [32]:
df_map.show(10, False)

+--------------------------+-----------+------+--------+---------+------------------------------------------+
|city                      |department |emp_id|name    |salary   |created_map                               |
+--------------------------+-----------+------+--------+---------+------------------------------------------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |{101 -> HR, HR -> Alice}                  |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |{102 -> Engineering, Engineering -> Bob}  |
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |{103 -> Sales, Sales -> Charlie}          |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |{104 -> Engineering, Engineering -> David}|
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |{105 -> Cleaner, Cleaner -> Charls}       |
|NULL                      |Saver      |106   |DeadPool|438438748|{106 -> Saver, Saver -> DeadPool}         |
+---------

In [ ]:
df_map.withColumn("exploded_map", explode(col("created_map")).alias("matrics_key", "matrics_value")).show(10,False)
## withColumn wont work here b/c of multiple map keys

In [34]:
df_map.select("created_map", explode(col("created_map")).alias("matrics_key", "matrics_value")).show(10,False)


+------------------------------------------+-----------+-------------+
|created_map                               |matrics_key|matrics_value|
+------------------------------------------+-----------+-------------+
|{101 -> HR, HR -> Alice}                  |101        |HR           |
|{101 -> HR, HR -> Alice}                  |HR         |Alice        |
|{102 -> Engineering, Engineering -> Bob}  |102        |Engineering  |
|{102 -> Engineering, Engineering -> Bob}  |Engineering|Bob          |
|{103 -> Sales, Sales -> Charlie}          |103        |Sales        |
|{103 -> Sales, Sales -> Charlie}          |Sales      |Charlie      |
|{104 -> Engineering, Engineering -> David}|104        |Engineering  |
|{104 -> Engineering, Engineering -> David}|Engineering|David        |
|{105 -> Cleaner, Cleaner -> Charls}       |105        |Cleaner      |
|{105 -> Cleaner, Cleaner -> Charls}       |Cleaner    |Charls       |
+------------------------------------------+-----------+-------------+
only s

## Working with JSON

- Spark has some unique support for working with JSON data. You can operate directly on strings
of JSON in Spark and parse from JSON or extract JSON objects.
- from_json : Converts a JSON string column into a structured StructType or MapType using a defined schema.
- to_json : Converts a StructType or MapType column back into a JSON-formatted string.

In [43]:
from pyspark.sql.functions import  col, to_json, get_json_object
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

df.show(10, False)


+--------------------------+-----------+------+--------+---------+
|city                      |department |emp_id|name    |salary   |
+--------------------------+-----------+------+--------+---------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |
|NULL                      |Saver      |106   |DeadPool|438438748|
+--------------------------+-----------+------+--------+---------+



In [56]:
from pyspark.sql.functions import col, struct

json_schema = StructType(
    [
    StructField("emp_id", IntegerType(), True),
    StructField("department", StringType(), True),
        StructField("city", StringType(), True),
]
)

struct_df = df.withColumn("complexType", struct("emp_id", "department", "city"))

parsed_df = (
    struct_df
    .withColumn("parsed_detail", to_json(col("complexType") ))## parsed_detail will be of stringtype
)

parsed_df.printSchema()

parsed_df.show(10, False)

root
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- complexType: struct (nullable = false)
 |    |-- emp_id: long (nullable = true)
 |    |-- department: string (nullable = true)
 |    |-- city: string (nullable = true)
 |-- parsed_detail: string (nullable = true)

+--------------------------+-----------+------+--------+---------+----------------------------------------------+-----------------------------------------------------------------------------+
|city                      |department |emp_id|name    |salary   |complexType                                   |parsed_detail                                                                |
+--------------------------+-----------+------+--------+---------+----------------------------------------------+-----------------------------------------------------------------------------+
|Pune, Banglor

In [ ]:
parsed_df.filter(col("parsed_detail.emp_id") == 101).show()  ## wont work

In [58]:
from pyspark.sql.functions import  from_json, to_json

## back to struct type
test = (parsed_df.select(from_json(col("parsed_detail"), json_schema).alias("json_struct")))
test.printSchema()
test.show(10, False)

root
 |-- json_struct: struct (nullable = true)
 |    |-- emp_id: integer (nullable = true)
 |    |-- department: string (nullable = true)
 |    |-- city: string (nullable = true)

+----------------------------------------------+
|json_struct                                   |
+----------------------------------------------+
|{101, HR, Pune, Banglore, Gorakhpur}          |
|{102, Engineering, PrayagRaj, Kolkata, Tuscon}|
|{103, Sales, Texax, LA, New Yark}             |
|{104, Engineering, Pune, New Yark, Gorakhpur} |
|{105, Cleaner, Pune, Aurangabad, None}        |
|{106, Saver, NULL}                            |
+----------------------------------------------+



In [60]:
test.filter(col("json_struct.emp_id") == 101).show() ## this would work

+--------------------+
|         json_struct|
+--------------------+
|{101, HR, Pune, B...|
+--------------------+



In [61]:
spark.stop()